# Formula 1 Podium Prediction Companion
**Milestone 1 | Reproducible pre-race analytics and prediction**

This notebook treats one `(raceId, driverId)` as one observation and predicts `podium = 1` when post-race `positionOrder <= 3`. That outcome is used only to create the label and to describe historical outcomes. Prediction time is after qualifying and final grid determination, before the race starts. Current-race results, points, status, laps, time, fastest lap, pit stops, and standings are excluded from predictors. Historical summaries are shifted before they are joined to an event.

Run every cell from top to bottom. Put the original Ergast-style CSV files anywhere beneath `DATA_ROOT`; files are read only. Generated outputs are saved under `outputs/`. The notebook sets all available random seeds to 42. Results are calculated from the discovered files; no example numbers are hard-coded.

## 0 — Configuration, reproducibility, and scope
**Purpose:** establish paths and a stable environment. **Method:** configure inputs, outputs, seeds, and model parameters once. **Assumptions:** source tables use Ergast-compatible columns. **Leakage risk:** configuration itself is not predictive; all feature allowlists are checked later. **Interpretation:** outputs depend on the exact CSV release discovered below.

In [3]:
from pathlib import Path
import os, re, sys, json, random, warnings, importlib.util, platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score, log_loss,
    precision_score, recall_score, confusion_matrix, balanced_accuracy_score,
    precision_recall_curve, roc_curve)
from sklearn.inspection import permutation_importance
from sklearn.base import clone

SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(SEED)
    TF_AVAILABLE = True
except Exception as exc:
    tf = None; TF_AVAILABLE = False
    print(f"TensorFlow unavailable ({exc}); the shallow neural network uses scikit-learn MLPClassifier and does not require TensorFlow.")
try:
    import shap
    SHAP_AVAILABLE = True
except Exception as exc:
    shap = None; SHAP_AVAILABLE = False
    print(f"SHAP unavailable ({exc}); install shap to enable SHAP explanations.")
try:
    import lime
    LIME_AVAILABLE = True
except Exception as exc:
    lime = None; LIME_AVAILABLE = False
    print(f"LIME unavailable ({exc}); local SHAP is attempted when possible.")

DATA_ROOT = Path(os.environ.get("F1_DATA_ROOT", Path.cwd()))  # environment override or notebook working folder
SEARCH_ROOTS = [DATA_ROOT]
if Path("/kaggle/input").exists() and Path("/kaggle/input").resolve() != DATA_ROOT.resolve(): SEARCH_ROOTS.append(Path("/kaggle/input"))
OUTPUT_ROOT = Path("outputs")
FIG_DIR, TABLE_DIR, ARTIFACT_DIR = OUTPUT_ROOT/"figures", OUTPUT_ROOT/"tables", OUTPUT_ROOT/"artifacts"
for p in (FIG_DIR, TABLE_DIR, ARTIFACT_DIR): p.mkdir(parents=True, exist_ok=True)
TRAIN_END_YEAR, VALIDATION_YEARS, TEST_YEARS = 2019, (2020, 2021), (2022, 2024)
MIN_GROUP_SIZE, MIN_FRONT_ROW_ENTRIES, MIN_CONSTRUCTOR_STARTS = 20, 10, 50
RF_TREES, RF_MAX_DEPTH, NN_EPOCHS, NN_BATCH_SIZE = 350, 12, 180, 256  # MLP maximum epochs and mini-batch size
print({"python":sys.version.split()[0],"platform":platform.platform(),"pandas":pd.__version__,
       "numpy":np.__version__,"sklearn":__import__('sklearn').__version__,"tensorflow_available":TF_AVAILABLE,
       "shap_available":SHAP_AVAILABLE,"lime_available":LIME_AVAILABLE,"seed":SEED,
       "data_roots_searched":[str(p.resolve()) for p in SEARCH_ROOTS],"output_root":str(OUTPUT_ROOT.resolve())})

TensorFlow unavailable (No module named 'tensorflow'); the shallow neural network uses scikit-learn MLPClassifier and does not require TensorFlow.
LIME unavailable (No module named 'lime'); local SHAP is attempted when possible.
{'python': '3.12.14', 'platform': 'Windows-11-10.0.26200-SP0', 'pandas': '3.0.6', 'numpy': '2.5.4', 'sklearn': '1.9.1', 'tensorflow_available': False, 'shap_available': True, 'lime_available': False, 'seed': 42, 'data_roots_searched': ['C:\\Users\\SPECTRE  X360  -14\\Documents\\Codex\\2026-10-14\\i-a\\work\\f1_dataset'], 'output_root': 'C:\\Users\\SPECTRE  X360  -14\\Documents\\Codex\\2026-10-14\\i-a\\outputs'}


## 1 — Recursive data discovery and loading
**Purpose:** find actual tables without assuming a folder layout. **Method:** recursively match normalized filename stems and retain paths/shapes/columns. **Assumptions:** CSVs are comma-separated and use conventional table names. **Leakage risk:** discovery loads forbidden post-race tables for auditing and label construction only; the feature allowlist later prevents their use. **Interpretation:** optional tables are reported, while missing core tables stop execution with a repair instruction.

In [5]:
TABLE_ALIASES = {
 "circuits":["circuits"], "constructors":["constructors"], "constructor_results":["constructor_results","constructorresults"],
 "constructor_standings":["constructor_standings","constructorstandings"], "drivers":["drivers"],
 "driver_standings":["driver_standings","driverstandings"], "races":["races"], "results":["results"],
 "qualifying":["qualifying"], "lap_times":["lap_times","laptimes"], "pit_stops":["pit_stops","pitstops"],
 "sprint_results":["sprint_results","sprintresults"], "seasons":["seasons"], "status":["status"]}
def norm_stem(path): return re.sub(r"[^a-z0-9]", "", path.stem.lower())
discovered = {}
for root in SEARCH_ROOTS:
    for p in root.rglob("*.csv"):
        stem = norm_stem(p)
        for name, aliases in TABLE_ALIASES.items():
            if stem in {re.sub(r"[^a-z0-9]", "", a) for a in aliases}:
                discovered.setdefault(name, []).append(p)
def read_table(paths):
    # If duplicate copies exist, select the largest file deterministically and log all candidates.
    p = sorted(paths, key=lambda x:(-x.stat().st_size, str(x).lower()))[0]
    return pd.read_csv(p, na_values=[r"\N", "", "NULL", "null", "N/A", "NA", "\\N"], keep_default_na=True, low_memory=False), p
raw_tables, source_paths = {}, {}
for name, paths in discovered.items():
    raw_tables[name], source_paths[name] = read_table(paths)
# Raw-data EDA BEFORE cleaning: preserve sentinels/types exactly as loaded and visualize coverage, missingness, and duplicate grain.
raw_eda_results=raw_tables["results"].copy()
raw_eda_races=raw_tables["races"][[c for c in ["raceId","year"] if c in raw_tables["races"]]].copy()
raw_eda_results["raceId"]=raw_eda_results["raceId"].astype(str)
raw_eda_races["raceId"]=raw_eda_races["raceId"].astype(str)
raw_eda=raw_eda_results.merge(raw_eda_races,on="raceId",how="left")
raw_eda["year_numeric"]=pd.to_numeric(raw_eda["year"],errors="coerce")
raw_eda_by_year=raw_eda.groupby("year_numeric",dropna=False).agg(raw_entries=("driverId","size"),raw_races=("raceId","nunique"),raw_duplicate_race_driver_rows=("driverId",lambda x: int(raw_eda.loc[x.index].duplicated(["raceId","driverId"],keep=False).sum()))).reset_index()
raw_eda_by_year.to_csv(TABLE_DIR/"raw_eda_by_season_before_cleaning.csv",index=False)
raw_missing=raw_eda_results.isna().mean().sort_values(ascending=False).rename("missing_fraction").reset_index().rename(columns={"index":"column"})
raw_missing.to_csv(TABLE_DIR/"raw_results_missingness_before_cleaning.csv",index=False)
fig,ax=plt.subplots(figsize=(11,4)); ax.plot(raw_eda_by_year.year_numeric,raw_eda_by_year.raw_entries,label="raw result rows"); ax.set(title="Raw race-result coverage by season (before cleaning)",xlabel="Season",ylabel="Rows"); ax.legend(); plt.tight_layout(); plt.savefig(FIG_DIR/"raw_entries_before_cleaning.png",dpi=160,bbox_inches="tight"); plt.show(); plt.close()
fig,ax=plt.subplots(figsize=(10,4)); sns.barplot(data=raw_missing.head(15),x="missing_fraction",y="column",color="#72b7b2",ax=ax); ax.set(title="Raw results missingness (before sentinel/type cleaning)",xlabel="Fraction parsed as missing by CSV reader",ylabel="Raw column",xlim=(0,1)); plt.tight_layout(); plt.savefig(FIG_DIR/"raw_missingness_before_cleaning.png",dpi=160,bbox_inches="tight"); plt.show(); plt.close()
fig,ax=plt.subplots(figsize=(11,4)); ax.plot(raw_eda_by_year.year_numeric,raw_eda_by_year.raw_duplicate_race_driver_rows,color="#e45756"); ax.set(title="Duplicate race-driver rows by season (before resolution)",xlabel="Season",ylabel="Rows in duplicate-key groups"); plt.tight_layout(); plt.savefig(FIG_DIR/"raw_duplicate_keys_before_cleaning.png",dpi=160,bbox_inches="tight"); plt.show(); plt.close()
print("Pre-cleaning EDA uses raw loaded values; paired post-cleaning charts and before/after row counts are in Section 3.")
inventory=[]
for name in TABLE_ALIASES:
    df=raw_tables.get(name)
    inventory.append({"table":name,"file_path":str(source_paths.get(name,"")),"shape":str(df.shape) if df is not None else "missing",
                      "columns":", ".join(df.columns.astype(str)) if df is not None else "", "missing_file":df is None,
                      "other_candidates":" | ".join(map(str,discovered.get(name,[])[1:]))})
inventory=pd.DataFrame(inventory); display(inventory); inventory.to_csv(TABLE_DIR/"data_inventory.csv",index=False)
REQUIRED_TABLES={"circuits","constructors","drivers","races","results","qualifying","status"}
missing=sorted(REQUIRED_TABLES-set(raw_tables))
if missing:
    raise FileNotFoundError("Missing required Ergast CSV table(s): " + ", ".join(missing) +
      f". Place the files anywhere under one of these searched roots: {[str(p.resolve()) for p in SEARCH_ROOTS]}. Expected names include circuits.csv, constructors.csv, drivers.csv, races.csv, results.csv, qualifying.csv, and status.csv.")
tables={k:v.copy(deep=True) for k,v in raw_tables.items()}
for n in ["races","results","qualifying","drivers","constructors","circuits"]:
    d=tables[n]; print(f"\\n{n}: shape={d.shape}"); display(d.head(3)); display(d.dtypes.to_frame("dtype")); display(d.isna().mean().sort_values(ascending=False).head(12).to_frame("missing_fraction"))

Pre-cleaning EDA uses raw loaded values; paired post-cleaning charts and before/after row counts are in Section 3.
                    table  ... other_candidates
0                circuits  ...                 
1            constructors  ...                 
2     constructor_results  ...                 
3   constructor_standings  ...                 
4                 drivers  ...                 
5        driver_standings  ...                 
6                   races  ...                 
7                 results  ...                 
8              qualifying  ...                 
9               lap_times  ...                 
10              pit_stops  ...                 
11         sprint_results  ...                 
12                seasons  ...                 
13                 status  ...                 

[14 rows x 6 columns]
\nraces: shape=(1125, 18)
   raceId  year  round  circuitId  ... quali_date quali_time sprint_date sprint_time
0       1  2009      1         

notebook_cell_4:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_4:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_4:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 2 — Data audit, cleaning, and entrant grain
**Purpose:** validate relational integrity and create one row per driver-race. **Method:** semantic conversions, key/FK audits, qualifying-time parsing, deterministic duplicate resolution. **Assumptions:** a nonzero grid represents a classified starting position; `grid=0` is retained and explicitly flagged because it can represent pit-lane or non-start cases. **Leakage risk:** results fields create the label and audit outcome; only pre-race columns survive into predictors. **Interpretation:** duplicate handling and missingness counts are saved for review; rows are not silently discarded.

In [7]:
cleaning_log=[]
def log_change(table, action, before, after, detail): cleaning_log.append({"table":table,"action":action,"rows_before":before,"rows_after":after,"detail":detail})
def numeric(df, cols, name):
    for c in cols:
        if c in df:
            b=len(df); df[c]=pd.to_numeric(df[c].replace({r"\\N":np.nan,"":np.nan}),errors="coerce"); log_change(name,f"numeric:{c}",b,len(df),f"nulls={df[c].isna().sum()}")
def q_seconds(x):
    if pd.isna(x): return np.nan
    s=str(x).strip()
    if not s or s in {r"\N", "\\N"}: return np.nan
    try:
        parts=s.split(":")
        if len(parts)==1: return float(parts[0])
        if len(parts)==2: return 60*float(parts[0])+float(parts[1])
        return np.nan
    except (ValueError,TypeError): return np.nan
for nm, cols in {"races":["year","round","raceId","circuitId"],"results":["raceId","driverId","constructorId","grid","positionOrder","resultId","statusId","points","laps"],"qualifying":["raceId","driverId","constructorId","position","qualifyId"],"circuits":["circuitId"],"drivers":["driverId"],"constructors":["constructorId"],"status":["statusId"],"driver_standings":["raceId","driverId","position","points"],"constructor_standings":["raceId","constructorId","position","points"]}.items():
    if nm in tables: numeric(tables[nm],cols,nm)
for nm, cols in {"races":["date"],"drivers":["dob"],"qualifying":[],"results":[]}.items():
    if nm in tables:
        for c in cols:
            if c in tables[nm]:
                old=tables[nm][c].isna().sum(); tables[nm][c]=pd.to_datetime(tables[nm][c],errors="coerce"); log_change(nm,f"datetime:{c}",len(tables[nm]),len(tables[nm]),f"new_nulls={tables[nm][c].isna().sum()-old}")
for c in ["q1","q2","q3"]:
    if c in tables["qualifying"]: tables["qualifying"][c+"_seconds"]=tables["qualifying"][c].map(q_seconds)
# Semantic typing for duration strings. Lap times are post-race and never model predictors; this conversion supports data audit/EDA only.
if "lap_times" in tables and "time" in tables["lap_times"]:
    tables["lap_times"]["time_seconds"]=tables["lap_times"]["time"].map(q_seconds)
    log_change("lap_times","duration:time_seconds",len(tables["lap_times"]),len(tables["lap_times"]),f"parseable={int(tables['lap_times']['time_seconds'].notna().sum())}; original time retained")
# results.time mixes elapsed times with text such as '+1 Lap'; only parse actual clock durations.
if "time" in tables["results"]:
    tables["results"]["time_seconds"]=tables["results"]["time"].map(q_seconds)
    log_change("results","duration:time_seconds",len(tables["results"]),len(tables["results"]),f"parseable={int(tables['results']['time_seconds'].notna().sum())}; non-duration labels remain missing")
key_specs={"circuits":["circuitId"],"constructors":["constructorId"],"drivers":["driverId"],"races":["raceId"],"results":["resultId"],"qualifying":["qualifyId"],"status":["statusId"]}
key_audit=[]
for nm,keys in key_specs.items():
    d=tables[nm]
    for c in keys:
        if c in d:
            key_audit.append({"table":nm,"key":c,"rows":len(d),"nulls":int(d[c].isna().sum()),"duplicate_rows":int(d[c].duplicated(keep=False).sum()),"present":True})
        else: key_audit.append({"table":nm,"key":c,"rows":len(d),"nulls":np.nan,"duplicate_rows":np.nan,"present":False})
key_audit=pd.DataFrame(key_audit); display(key_audit); key_audit.to_csv(TABLE_DIR/"key_audit.csv",index=False)
fk_specs=[("races","circuitId","circuits","circuitId"),("results","raceId","races","raceId"),("results","driverId","drivers","driverId"),("results","constructorId","constructors","constructorId"),("results","statusId","status","statusId"),("qualifying","raceId","races","raceId"),("qualifying","driverId","drivers","driverId"),("qualifying","constructorId","constructors","constructorId")]
for a in ["driver_standings","constructor_standings"]:
    if a in tables:
        fk_specs += [(a,"raceId","races","raceId"),(a,"driverId" if a=="driver_standings" else "constructorId","drivers" if a=="driver_standings" else "constructors","driverId" if a=="driver_standings" else "constructorId")]
fk=[]
for child,col,parent,pcol in fk_specs:
    if col in tables[child] and pcol in tables[parent]:
        orphan=~tables[child][col].isin(tables[parent][pcol]); fk.append({"child":child,"column":col,"parent":parent,"parent_key":pcol,"orphan_rows":int(orphan.sum()),"null_rows":int(tables[child][col].isna().sum())})
fk=pd.DataFrame(fk); display(fk); fk.to_csv(TABLE_DIR/"foreign_key_audit.csv",index=False)
res=tables["results"].copy(); races=tables["races"].copy()
if "positionOrder" not in res: raise KeyError("results.csv must contain positionOrder to construct the required podium target.")
if "grid" not in res: raise KeyError("results.csv must contain grid to define the final starting position at the pre-race cutoff.")
res["podium"]=(res["positionOrder"]<=3).astype("Int64"); dup=res[res.duplicated(["raceId","driverId"],keep=False)].copy()
if not dup.empty:
    dup=dup.merge(races[["raceId","year"]],on="raceId",how="left"); display(dup.sort_values(["year","raceId","driverId"])); dup.groupby("year").size().rename("duplicate_rows").to_csv(TABLE_DIR/"duplicate_rows_by_year.csv")
    res["_grid_nonzero"]=res["grid"].fillna(0).ne(0); res["_result_order"]=res["positionOrder"].fillna(np.inf); res["_result_id"]=res.get("resultId",pd.Series(np.inf,index=res.index)).fillna(np.inf)
    res=res.sort_values(["_grid_nonzero","_result_order","_result_id"],ascending=[False,True,True],kind="mergesort").drop_duplicates(["raceId","driverId"],keep="first").drop(columns=["_grid_nonzero","_result_order","_result_id"])
assert not res.duplicated(["raceId","driverId"]).any(),"(raceId, driverId) must be unique after deterministic duplicate resolution."
res=res.merge(races[[c for c in ["raceId","year","round","circuitId","date","name"] if c in races]],on="raceId",how="left",validate="many_to_one",suffixes=("","_race"))
res["grid_zero"]=res.get("grid",pd.Series(np.nan,index=res.index)).fillna(0).eq(0)
res["grid_zero_or_unknown"]=res["grid_zero"].astype(int)
res["qualifying_missing"]=~res["raceId"].astype(str).str.cat(res["driverId"].astype(str),sep="_").isin(tables["qualifying"]["raceId"].astype(str).str.cat(tables["qualifying"]["driverId"].astype(str),sep="_"))
tables["clean_results"]=res
log_change("results","resolve duplicate race-driver rows",len(tables["results"]),len(res),"prefer nonzero grid; then lowest positionOrder; then smallest resultId. grid=0/missing qualifying retained and flagged.")
before_raw=tables["results"]
clean_comparison=pd.DataFrame([
 {"stage":"raw results","rows":len(before_raw),"races":before_raw.raceId.nunique(),"drivers":before_raw.driverId.nunique(),"constructors":before_raw.constructorId.nunique(),"seasons":races.loc[races.raceId.isin(before_raw.raceId),"year"].nunique()},
 {"stage":"cleaned race-driver entries","rows":len(res),"races":res.raceId.nunique(),"drivers":res.driverId.nunique(),"constructors":res.constructorId.nunique(),"seasons":res.year.nunique()}])
clean_comparison.to_csv(TABLE_DIR/"before_after_cleaning.csv",index=False); display(clean_comparison)
cleanlog=pd.DataFrame(cleaning_log); display(cleanlog); cleanlog.to_csv(TABLE_DIR/"cleaning_log.csv",index=False)
print({"grid_zero_rows":int(res["grid_zero"].sum()),"missing_qualifying_rows":int(res["qualifying_missing"].sum()),"duplicate_race_driver_groups":int(dup.groupby(["raceId","driverId"]).ngroups if not dup.empty else 0),"withdrawals_or_nonstarters_retained":int((res["grid"].fillna(0)==0).sum())})

          table            key   rows  nulls  duplicate_rows  present
0      circuits      circuitId     77      0               0     True
1  constructors  constructorId    212      0               0     True
2       drivers       driverId    861      0               0     True
3         races         raceId   1125      0               0     True
4       results       resultId  26759      0               0     True
5    qualifying      qualifyId  10494      0               0     True
6        status       statusId    139      0               0     True
                    child         column  ... orphan_rows null_rows
0                   races      circuitId  ...           0         0
1                 results         raceId  ...           0         0
2                 results       driverId  ...           0         0
3                 results  constructorId  ...           0         0
4                 results       statusId  ...           0         0
5              qualifying       

## 3 — Exploratory analysis before and after cleaning
**Purpose:** quantify source coverage, missingness, duplicate grain, and how cleaning changes the data. **Method:** Section 1 visualizes raw loaded values before semantic conversion and duplicate resolution; this section pairs raw and cleaned season counts and plots cleaned coverage, missingness, grid, and podium patterns. **Assumptions:** raw sentinels are treated as file values until the cleaning step; podium labels use `positionOrder`. **Leakage risk:** outcomes appear only in descriptive analysis, never as model predictors. **Interpretation:** historical coverage and class balance vary by era; raw/clean differences are saved for audit.

In [9]:
def savefig(name):
    plt.tight_layout(); plt.savefig(FIG_DIR/name,dpi=160,bbox_inches="tight"); plt.show(); plt.close()
season=res.groupby("year").agg(entries=("driverId","size"),races=("raceId","nunique"),drivers=("driverId","nunique"),podium_rate=("podium","mean"),qualifying_missing_rate=("qualifying_missing","mean"),zero_grid_rate=("grid_zero","mean")).reset_index()
season.to_csv(TABLE_DIR/"season_coverage.csv",index=False); display(season.tail(15))
# Direct before/after cleaning comparison at the season grain.
clean_by_year=season[["year","entries"]].rename(columns={"entries":"clean_entries"})
impact=raw_eda_by_year.rename(columns={"year_numeric":"year","raw_entries":"raw_entries","raw_duplicate_race_driver_rows":"raw_duplicate_rows"}).merge(clean_by_year,on="year",how="outer").fillna(0).sort_values("year")
impact["rows_removed_by_duplicate_resolution"]=impact.raw_entries-impact.clean_entries
impact.to_csv(TABLE_DIR/"cleaning_impact_by_season.csv",index=False); display(impact.tail(15))
fig,ax=plt.subplots(figsize=(11,4)); ax.plot(impact.year,impact.raw_entries,label="raw rows",alpha=.8); ax.plot(impact.year,impact.clean_entries,label="clean unique driver-race rows",alpha=.8); ax.set(title="Race-entry counts before vs after cleaning",xlabel="Season",ylabel="Rows"); ax.legend(); savefig("cleaning_impact_by_season.png")
fig,ax=plt.subplots(figsize=(11,4)); ax.plot(season.year,season.entries,label="entries"); ax.plot(season.year,season.races,label="races"); ax.set(title="Race and entrant coverage by season",xlabel="Season",ylabel="Count"); ax.legend(); savefig("coverage_by_season.png")
fig,ax=plt.subplots(figsize=(10,4)); (1-res[["grid","qualifying_missing"]].isna().mean()).plot.bar(ax=ax); ax.set(title="Availability of core pre-race fields",ylabel="Fraction observed",ylim=(0,1)); savefig("core_field_coverage.png")
fig,ax=plt.subplots(figsize=(10,4)); season.plot(x="year",y="qualifying_missing_rate",ax=ax,legend=False); ax.set(title="Missing qualifying observation by season",xlabel="Season",ylabel="Missing fraction",ylim=(0,1)); savefig("qualifying_missing_by_season.png")
fig,ax=plt.subplots(figsize=(10,4)); sns.histplot(data=res[res.grid>0],x="grid",bins=np.arange(0,45,1),ax=ax); ax.set(title="Distribution of known grid positions",xlabel="Grid position",ylabel="Race entries"); savefig("grid_distribution.png")
fig,ax=plt.subplots(figsize=(12,4)); sns.lineplot(data=season,x="year",y="podium_rate",marker="o",ax=ax); ax.set(title="Observed podium rate by season",xlabel="Season",ylabel="Podium fraction"); savefig("podium_rate_by_season.png")
grid_rate=res[res.grid.between(1,30)].groupby("grid").agg(entries=("podium","size"),podium_rate=("podium","mean")).query("entries >= @MIN_GROUP_SIZE").reset_index()
fig,ax=plt.subplots(figsize=(10,4)); sns.barplot(data=grid_rate,x="grid",y="podium_rate",color="#377eb8",ax=ax); ax.set(title="Historical podium rate by starting grid",xlabel="Grid position",ylabel="Podium fraction"); savefig("podium_rate_by_grid.png")
fig,ax=plt.subplots(figsize=(8,4)); sns.countplot(data=res,x="podium",ax=ax); ax.set(title="Overall podium class balance",xlabel="Podium label (0/1)",ylabel="Entrants"); savefig("class_balance.png")
fig,ax=plt.subplots(figsize=(10,4)); season.plot(x="year",y="entries",kind="bar",ax=ax,legend=False); ax.set(title="Entrants per season (race-entry total)",xlabel="Season",ylabel="Entries"); savefig("entries_by_season.png")
per_race=res.groupby(["year","raceId"]).driverId.nunique().groupby("year").agg(["mean","median","min","max"]).reset_index(); per_race.to_csv(TABLE_DIR/"entrants_per_race_by_season.csv",index=False)
fig,ax=plt.subplots(figsize=(10,4)); ax.plot(per_race.year,per_race["mean"],marker="o",label="mean"); ax.plot(per_race.year,per_race["median"],marker=".",label="median"); ax.set(title="Entrants per race by season",xlabel="Season",ylabel="Drivers per race"); ax.legend(); savefig("entrants_per_race_by_season.png")
class_time=res.groupby(["year","podium"]).size().rename("entries").reset_index()
fig,ax=plt.subplots(figsize=(10,4)); sns.lineplot(data=class_time,x="year",y="entries",hue="podium",marker="o",ax=ax); ax.set(title="Class counts over time",xlabel="Season",ylabel="Race entries"); savefig("class_balance_by_season.png")
print("Interpretation: use the displayed counts and rates only; small groups and incomplete qualifying coverage limit comparisons.")

    year  entries  races  ...  podium_rate  qualifying_missing_rate  zero_grid_rate
60  2010      456     19  ...        0.125                 0.000000        0.000000
61  2011      456     19  ...        0.125                 0.008772        0.006579
62  2012      480     20  ...        0.125                 0.008333        0.006250
63  2013      418     19  ...     0.136364                 0.000000        0.000000
64  2014      407     19  ...     0.140049                 0.004914        0.000000
65  2015      378     19  ...     0.150794                 0.010582        0.007937
66  2016      462     21  ...     0.136364                 0.010823        0.008658
67  2017      400     20  ...         0.15                 0.005000        0.005000
68  2018      420     21  ...         0.15                 0.000000        0.002381
69  2019      420     21  ...         0.15                 0.004762        0.028571
70  2020      340     17  ...         0.15                 0.000000        0

notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 4 — Required data-engineering questions
**Purpose:** answer the three historical questions at their stated grain. **Method:** join results through races and entity tables; save group counts, rates, and charts. Q1 ranks actual starting-grid positions 1–2 and separately audits where qualifying position differs from grid. Q2 reports the all-race estimate and an Indianapolis 500 sensitivity excluding 1950–1960. **Assumptions:** minimum-size thresholds are displayed; Indianapolis 500 was a distinct oval event. **Leakage risk:** these are retrospective descriptive questions; outcomes are not model features. **Interpretation:** all rates are descriptive, not causal.

In [11]:
# Q1: grid 1/2, explicitly use starting grid. Ranking uses period + circuit; show sample size.
q1=res[res.grid.isin([1,2])].copy(); q1["period"]=np.select([q1.year<2014,q1.year.between(2014,2021),q1.year.between(2022,2024)],["pre-2014","2014-2021","2022-2024"],default="outside comparison")
qual_audit=tables["qualifying"].sort_values([c for c in ["raceId","driverId","position"] if c in tables["qualifying"]]).drop_duplicates(["raceId","driverId"],keep="first")[[c for c in ["raceId","driverId","position"] if c in tables["qualifying"]]].rename(columns={"position":"qualifying_position"})
q1_grid_audit=res.merge(qual_audit,on=["raceId","driverId"],how="left"); q1_grid_audit["grid_differs_from_qualifying"]=q1_grid_audit.grid.notna() & q1_grid_audit.qualifying_position.notna() & q1_grid_audit.grid.ne(q1_grid_audit.qualifying_position)
q1_grid_audit["has_both_positions"]=q1_grid_audit.grid.notna() & q1_grid_audit.qualifying_position.notna(); q1_grid_audit_summary=q1_grid_audit.groupby("year").agg(entries_with_both=("has_both_positions","sum"),grid_qualifying_differences=("grid_differs_from_qualifying","sum")).reset_index(); q1_grid_audit_summary["difference_rate_among_entries_with_both"]=q1_grid_audit_summary.grid_qualifying_differences/q1_grid_audit_summary.entries_with_both.replace(0,np.nan)
q1_grid_audit_summary.to_csv(TABLE_DIR/"q1_grid_vs_qualifying_audit.csv",index=False); display(q1_grid_audit_summary.tail(15))
print("Q1 uses actual starting grid because the question asks about starts; qualifying rank is distinct and may differ after penalties/pit-lane starts.")
if "circuits" in tables:
    q1=q1.merge(tables["circuits"][[c for c in ["circuitId","name","country"] if c in tables["circuits"]]],on="circuitId",how="left",suffixes=("","_circuit"))
q1name="name_circuit" if "name_circuit" in q1 else "name"
q1tab=q1.groupby(["period","circuitId",q1name],dropna=False).agg(front_row_entries=("podium","size"),podiums=("podium","sum"),conversion_rate=("podium","mean")).reset_index().rename(columns={q1name:"circuit"})
q1tab["eligible_for_rank"]=q1tab.front_row_entries>=MIN_FRONT_ROW_ENTRIES; q1tab["rank"]=q1tab[q1tab.eligible_for_rank].groupby("period").conversion_rate.rank(method="min",ascending=False); q1tab.to_csv(TABLE_DIR/"q1_front_row_conversion.csv",index=False); display(q1tab.sort_values(["period","rank"],na_position="last")); print(f"Q1 ranking threshold: {MIN_FRONT_ROW_ENTRIES} front-row entries per circuit-period.")
for period,g in q1tab[q1tab.eligible_for_rank].groupby("period"):
    g=g.nlargest(12,"conversion_rate"); plt.figure(figsize=(10,5)); sns.barplot(data=g,x="conversion_rate",y="circuit",color="#4c78a8"); plt.title(f"Front-row podium conversion by circuit: {period}"); plt.xlabel("Podium fraction (front-row entries)"); plt.ylabel("Circuit"); savefig(f"q1_front_row_{period.replace('-','_')}.png")

# Q2: explicit normalized mapping; unknown mappings are retained as unmatched and shown.
country_map={"UK":"United Kingdom","U.K.":"United Kingdom","Britain":"United Kingdom","British":"United Kingdom","USA":"United States","United States of America":"United States","American":"United States","Italian":"Italy","French":"France","German":"Germany","East German":"Germany","Spanish":"Spain","Dutch":"Netherlands","Australian":"Australia","Canadian":"Canada","Brazilian":"Brazil","Mexican":"Mexico","Japanese":"Japan","Monegasque":"Monaco","Swiss":"Switzerland","Austrian":"Austria","Belgian":"Belgium","Finnish":"Finland","Danish":"Denmark","Swedish":"Sweden","Argentine":"Argentina","Argentinian":"Argentina","New Zealander":"New Zealand","Thai":"Thailand","Chinese":"China","Russian":"Russia","Polish":"Poland","Indian":"India","South African":"South Africa","Irish":"Ireland","Portuguese":"Portugal","Uruguayan":"Uruguay","Colombian":"Colombia","Venezuelan":"Venezuela","Hungarian":"Hungary","Czech":"Czechia","Malaysian":"Malaysia","Chilean":"Chile","Liechtensteiner":"Liechtenstein","Indonesian":"Indonesia"}
country_mapping_table=pd.DataFrame([{"source_value":k,"normalized_country":v} for k,v in country_map.items()]); country_mapping_table.to_csv(TABLE_DIR/"country_normalization_mapping.csv",index=False); display(country_mapping_table)
def canon(x):
    if pd.isna(x): return np.nan
    s=str(x).strip(); return country_map.get(s,s)
q2=res.merge(races[[c for c in ["raceId","circuitId","year"] if c in races]],on="raceId",how="left",suffixes=("","_race"))
q2=q2.merge(tables["circuits"][[c for c in ["circuitId","country"] if c in tables["circuits"]]],on="circuitId",how="left")
indy_ids=set(tables["circuits"].loc[tables["circuits"]["name"].astype(str).str.contains("Indianapolis",case=False,na=False),"circuitId"].tolist()) if "name" in tables["circuits"] else set()
q2["is_indianapolis_500"]=q2.circuitId.isin(indy_ids) & q2.year.between(1950,1960)
q2=q2.merge(tables["drivers"][[c for c in ["driverId","nationality"] if c in tables["drivers"]]],on="driverId",how="left")
q2["race_country_norm"]=q2.country.map(canon); q2["driver_country_norm"]=q2.nationality.map(canon)
known_countries=set(q2.race_country_norm.dropna().unique())|set(country_map.values())
q2["country_unmatched"]=q2.country.isna()|(q2.nationality.notna() & ~q2.nationality.isin(country_map) & ~q2.driver_country_norm.isin(known_countries))|q2.nationality.isna()
q2["is_home_race"]=q2.race_country_norm.eq(q2.driver_country_norm).where(~q2.country_unmatched,pd.NA)
q2["grid_bin"]=pd.cut(q2.grid.where(q2.grid>0),bins=[0,2,5,10,np.inf],labels=["1-2","3-5","6-10","11+"],include_lowest=True).astype("object"); q2.loc[q2.grid.fillna(0).eq(0),"grid_bin"]="0 / unknown"
unmatched=q2.loc[q2.country_unmatched,["country","nationality"]].value_counts(dropna=False).rename("entries").reset_index(); unmatched.to_csv(TABLE_DIR/"q2_unmatched_countries.csv",index=False); display(unmatched.head(30))
q2tab=q2.groupby(["grid_bin","is_home_race"],dropna=False).agg(entries=("podium","size"),podiums=("podium","sum"),podium_rate=("podium","mean")).reset_index(); q2tab.to_csv(TABLE_DIR/"q2_home_by_grid.csv",index=False); display(q2tab)
q2valid=q2[~q2.country_unmatched].copy(); q2overall=q2valid.groupby("is_home_race").agg(entries=("podium","size"),rate=("podium","mean")); weights=q2valid.groupby("grid_bin").size()/len(q2valid); bybin=q2valid.groupby(["grid_bin","is_home_race"]).podium.mean().unstack(); common=bybin.dropna().index; common_weights=weights.reindex(common); common_weights=common_weights/common_weights.sum() if common_weights.sum()>0 else common_weights; standardized={k:float((bybin.loc[common,k]*common_weights).sum()) for k in bybin.columns}; print("Grid-bin-weighted descriptive rates on shared support (unmatched country values excluded):",standardized,"; difference(home-away):",standardized.get(True,np.nan)-standardized.get(False,np.nan),"; bins:",list(common))
fig,ax=plt.subplots(figsize=(9,5)); sns.pointplot(data=q2tab,x="grid_bin",y="podium_rate",hue="is_home_race",ax=ax); ax.set(title="Home-country podium rates within starting-grid bins",xlabel="Starting grid bin",ylabel="Podium fraction"); savefig("q2_home_grid_rates.png")
# Indianapolis 500 sensitivity: the event is present in championship tables but used a distinct oval/entry context.
def standardized_home_rates(frame):
    v=frame[~frame.country_unmatched].copy()
    w=v.groupby("grid_bin").size()/len(v)
    b=v.groupby(["grid_bin","is_home_race"]).podium.mean().unstack()
    common=b.dropna().index; w=w.reindex(common); w=w/w.sum() if w.sum()>0 else w
    rates={k:float((b.loc[common,k]*w).sum()) for k in b.columns}
    return rates,int(len(v)),list(common)
indy_sensitivity=[]
for label,frame in [("all championship races",q2),("excluding Indianapolis 500 (1950-1960)",q2[~q2.is_indianapolis_500])]:
    rates,n,common_bins=standardized_home_rates(frame)
    indy_sensitivity.append({"scope":label,"entries":n,"home_rate":rates.get(True,np.nan),"away_rate":rates.get(False,np.nan),"difference_home_minus_away":rates.get(True,np.nan)-rates.get(False,np.nan),"common_grid_bins":";".join(map(str,common_bins))})
indy_sensitivity=pd.DataFrame(indy_sensitivity); indy_sensitivity.to_csv(TABLE_DIR/"q2_indianapolis_sensitivity.csv",index=False); display(indy_sensitivity)
print("Primary Q2 includes all championship races for completeness; the sensitivity row excludes Indianapolis 500 races from 1950-1960 because their oval/entry structure differs.")

# Q3: conservative status-text mapping. Denominator excludes entries explicitly labeled as not started.
q3=res.merge(tables["constructors"][[c for c in ["constructorId","name"] if c in tables["constructors"]]],on="constructorId",how="left",suffixes=("","_constructor"))
q3=q3.merge(tables["status"][[c for c in ["statusId","status"] if c in tables["status"]]],on="statusId",how="left")
MECH_PAT=r"engine|gearbox|transmission|hydraulic|brake|suspension|electrical|fuel|power unit|turbo|battery|cooling|water|oil|clutch|radiator|alternator|spark plug"
EXCLUDE_PAT=r"collision|accident|crash|spun|injur|driver|disqualif|excluded|regulation|withdraw|did not start|not started"
def status_class(s):
    x="" if pd.isna(s) else str(s).lower()
    if re.search(EXCLUDE_PAT,x): return "excluded/non-mechanical"
    if re.search(MECH_PAT,x): return "mechanical"
    if x in {"finished"," +1 lap","+1 lap","+2 laps","+3 laps","+4 laps","+5 laps"} or "lap" in x: return "finished/classified"
    return "other/ambiguous"
q3["status_class"]=q3.status.map(status_class); q3["period"]=np.select([q3.year.between(2014,2021),q3.year.between(2022,2024)],["2014-2021","2022-2024"],default="outside comparison")
not_started_pat=r"did not start|not started|withdraw|\bdns\b|didn't start|failed to start|did not qualify|not qualified"
q3["started_proxy"]=~q3.status.astype(str).str.lower().str.contains(not_started_pat,regex=True,na=False)
status_map=q3[["status","status_class"]].drop_duplicates().sort_values(["status_class","status"]); status_map.to_csv(TABLE_DIR/"q3_status_mapping.csv",index=False); display(status_map)
status_counts=q3.groupby("status_class",dropna=False).size().rename("entries").reset_index(); status_counts.to_csv(TABLE_DIR/"q3_status_class_counts.csv",index=False); display(status_counts)
q3["mechanical_retirement"]=q3.status_class.eq("mechanical") & ~q3.status.astype(str).str.lower().isin(["finished"])
constructor_name="name_constructor" if "name_constructor" in q3.columns else "name"
q3tab=q3[q3.period.isin(["2014-2021","2022-2024"]) & q3.started_proxy].groupby(["period","constructorId",constructor_name],dropna=False).agg(starts=("raceId","size"),mechanical_retirements=("mechanical_retirement","sum")).reset_index().rename(columns={constructor_name:"constructor"}); q3tab["mechanical_rate"]=q3tab.mechanical_retirements/q3tab.starts; q3tab["eligible_for_rank"]=q3tab.starts>=MIN_CONSTRUCTOR_STARTS; q3tab["rank"]=q3tab[q3tab.eligible_for_rank].groupby("period").mechanical_rate.rank(method="min",ascending=False); q3tab.to_csv(TABLE_DIR/"q3_mechanical_retirements.csv",index=False); display(q3tab.sort_values(["period","rank"],na_position="last"))
for period,g in q3tab[q3tab.eligible_for_rank].groupby("period"):
    g=g.nlargest(12,"mechanical_rate"); plt.figure(figsize=(10,5)); sns.barplot(data=g,x="mechanical_rate",y="constructor",color="#e45756"); plt.title(f"Mechanical-retirement rate by constructor: {period}"); plt.xlabel("Mechanical retirements / starts"); plt.ylabel("Constructor"); savefig(f"q3_mechanical_{period.replace('-','_')}.png")

    year  ...  difference_rate_among_entries_with_both
60  2010  ...                                 0.201754
61  2011  ...                                 0.176991
62  2012  ...                                 0.481092
63  2013  ...                                 0.294258
64  2014  ...                                 0.355556
65  2015  ...                                 0.334225
66  2016  ...                                 0.273523
67  2017  ...                                 0.409548
68  2018  ...                                 0.264286
69  2019  ...                                 0.385167
70  2020  ...                                 0.179412
71  2021  ...                                 0.362187
72  2022  ...                                 0.377273
73  2023  ...                                 0.279545
74  2024  ...                                 0.242171

[15 rows x 4 columns]
Q1 uses actual starting grid because the question asks about starts; qualifying rank is distinct 

notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Required question answers (computed from this dataset release)

### Q1. Front-row podium conversion across circuits and eras
The front row is defined by actual starting grid positions 1 and 2, not qualifying rank. Qualifying rank and final grid may differ because of penalties or pit-lane starts; `q1_grid_vs_qualifying_audit.csv` quantifies those differences. Circuit-period rates are descriptive and ranked only when a circuit has at least 10 front-row entries. In the 2014-2021 comparison, the leading eligible circuit is shown in `q1_front_row_conversion.csv` and the companion chart; the 2022-2024 period has too few seasons to support a robust cross-era conclusion under the same threshold. The pre-2014 period has broader historical coverage but combines changing cars, rules, and calendars. These rates describe observed conversion and are not causal estimates.

### Q2. Home-country podium rate at comparable grid positions
Home and away entrants are compared within the same grid-position bins so the result is not simply a comparison of different starting positions. On the common supported grid bins, the weighted descriptive podium rate is **12.44% for home entrants** and **12.77% for away entrants** (home minus away: **-0.32 percentage points**). Several nationality/country labels do not normalize cleanly and are listed in `q2_unmatched_countries.csv`. The Indianapolis 500 ran as a championship round through 1960 but had a distinct oval and entry context. The primary estimate retains the full championship dataset; `q2_indianapolis_sensitivity.csv` reports the corresponding estimate excluding those races. Small counts and historical era effects mean this should not be interpreted as evidence that home races help or hurt drivers.

### Q3. Constructor mechanical-retirement rates
Mechanical retirements are identified from a documented keyword mapping of status labels and divided by constructor starts; only constructors with at least 50 starts are ranked. For 2014-2021, Toro Rosso has the highest eligible rate (35 of 242 starts; **14.46%**), followed by Lotus F1 (9/76; **11.84%**) and Renault (23/200; **11.50%**). For 2022-2024, Alpine has the highest eligible mapped rate (13/136; **9.56%**). Status wording is imperfect, so these are estimates under the stated mapping rather than ground-truth engineering failure rates.


## 5 — Leakage-safe modeling table and feature dictionary
**Purpose:** build a pre-race prediction table with historical form. **Method:** join qualifying/current-event metadata and compute driver/constructor histories by chronological `shift(1)` then rolling/expanding summaries. Previous standings are shifted within season and joined by prior race. **Assumptions:** a missing historical feature means no recorded earlier experience, not zero ability. **Leakage risk:** source results are label/outcome-only except prior events; only the pre-race qualifying roster determines field size; assertions reject forbidden feature names. Sprint results are excluded because the source does not guarantee they are timestamped before the main race; the recorded final `grid` is the usable pre-start order where available. Indianapolis 500 rows are retained for championship completeness and flagged for descriptive sensitivity checks. **Interpretation:** high-cardinality IDs are retained only as categorical context and use one-hot encoding with unknown-category handling; the model's extrapolation to unseen drivers/constructors is limited.

In [14]:
# Qualifying table can have duplicates; choose the best qualifying row per race-driver by lowest numeric position.
qual=tables["qualifying"].copy(); qual=qual.sort_values([c for c in ["raceId","driverId","position"] if c in qual]).drop_duplicates(["raceId","driverId"],keep="first")
qcols=[c for c in ["raceId","driverId","position","q1_seconds","q2_seconds","q3_seconds"] if c in qual]
model=res.merge(qual[qcols],on=["raceId","driverId"],how="left",validate="one_to_one")
model=model.merge(tables["drivers"][[c for c in ["driverId","nationality"] if c in tables["drivers"]]],on="driverId",how="left")
model=model.merge(tables["constructors"][[c for c in ["constructorId","nationality"] if c in tables["constructors"]]],on="constructorId",how="left",suffixes=("_driver","_constructor"))
model=model.merge(tables["circuits"][[c for c in ["circuitId","country"] if c in tables["circuits"]]],on="circuitId",how="left")
model["is_home_race"]=(model.country.map(canon)==model.nationality_driver.map(canon)).astype("Int64")
qtime_cols=[c for c in ["q1_seconds","q2_seconds","q3_seconds"] if c in model]
if qtime_cols:
    model["best_qualifying_seconds"]=model[[c for c in ["q3_seconds","q2_seconds","q1_seconds"] if c in model]].bfill(axis=1).iloc[:,0]
    model["qualifying_gap_to_pole"]=np.nan
    for rid, ix in model.groupby("raceId").groups.items():
        for col in ["q3_seconds","q2_seconds","q1_seconds"]:
            if col in model and model.loc[ix,col].notna().any():
                pole_time=model.loc[ix,col].min(); model.loc[ix,"qualifying_gap_to_pole"]=model.loc[ix,col]-pole_time; break
model["qualifying_available"]=model[[c for c in ["q1_seconds","q2_seconds","q3_seconds"] if c in model]].notna().any(axis=1).astype(int)
qualifying_field_size=qual.groupby("raceId").driverId.nunique()
model["field_size"]=model.raceId.map(qualifying_field_size)
model=model.sort_values(["date","year","round","raceId"],kind="mergesort")
def add_history(frame, entity, prefix):
    frame=frame.sort_values(["date","year","round","raceId"],kind="mergesort").copy()
    # Aggregate to one entity-event before shifting. This is essential for constructors,
    # which have multiple driver rows in the same race; otherwise the first teammate's
    # current-race outcome would leak into the second teammate's features.
    event=frame.groupby([entity,"raceId"],dropna=False).agg(
        event_podium=("podium","max"),event_finish=("positionOrder","mean"),
        date=("date","first"),year=("year","first"),round=("round","first")).reset_index()
    event=event.sort_values([entity,"date","year","round","raceId"],kind="mergesort")
    g=event.groupby(entity,sort=False,dropna=False)
    event[prefix+"_prior_race_count"]=g.cumcount()
    event[prefix+"_prior_podium_rate_expanding"]=g.event_podium.transform(lambda s:s.shift(1).expanding().mean())
    event[prefix+"_prior_finish_mean_expanding"]=g.event_finish.transform(lambda s:s.shift(1).expanding().mean())
    event[prefix+"_prior_podium_rate_last3"]=g.event_podium.transform(lambda s:s.shift(1).rolling(3,min_periods=1).mean())
    event[prefix+"_prior_podium_rate_last5"]=g.event_podium.transform(lambda s:s.shift(1).rolling(5,min_periods=1).mean())
    histcols=[c for c in event if c.startswith(prefix+"_prior_")]
    return frame.merge(event[[entity,"raceId"]+histcols],on=[entity,"raceId"],how="left",validate="many_to_one")
model=add_history(model,"driverId","driver")
model=add_history(model,"constructorId","constructor")
constructor_hist=[c for c in model if c.startswith("constructor_prior_")]
assert not model.groupby(["raceId","constructorId"])[constructor_hist].nunique(dropna=False).gt(1).any().any(),"Constructor history must be constant across teammates within each race."
for entity,prefix in [("driverId","driver"),("constructorId","constructor")]:
    first=model.sort_values(["date","year","round","raceId"]).groupby(entity,dropna=False).head(1)
    assert first[prefix+"_prior_race_count"].fillna(0).eq(0).all(),f"First observed {entity} event must have zero prior-event count."
    assert first[prefix+"_prior_podium_rate_expanding"].isna().all(),f"First observed {entity} event cannot have a prior podium rate."
    display(first[[entity,"raceId",prefix+"_prior_race_count",prefix+"_prior_podium_rate_expanding"]].head(5))
# Perturb one current outcome and confirm the same event's history stays unchanged.
for entity,prefix in [("driverId","driver"),("constructorId","constructor")]:
    probe_cols=[entity,"raceId","date","year","round","podium","positionOrder"]
    probe=model[probe_cols].copy(); base=add_history(probe,entity,prefix)
    candidate=model.sort_values(["date","year","round","raceId"]).iloc[-1]
    key_entity,key_race=candidate[entity],candidate["raceId"]
    changed_probe=probe.copy(); mask=(changed_probe[entity]==key_entity)&(changed_probe.raceId==key_race)
    changed_probe.loc[mask,"podium"]=1-changed_probe.loc[mask,"podium"].astype(int)
    changed_probe.loc[mask,"positionOrder"]=np.nan
    changed=add_history(changed_probe,entity,prefix)
    check_cols=[c for c in base if c.startswith(prefix+"_prior_")]
    assert base.loc[mask,check_cols].reset_index(drop=True).equals(changed.loc[mask,check_cols].reset_index(drop=True)),f"Current {entity} event outcome affected its own history features."
# Shift standings by one event within season before attaching to the current event.
for name, entity, prefix in [("driver_standings","driverId","driver"),("constructor_standings","constructorId","constructor")]:
    if name in tables:
        st=tables[name].merge(races[[c for c in ["raceId","year","round","date"] if c in races]],on="raceId",how="left")
        st=st.sort_values([entity,"year","date","round"],kind="mergesort")
        for col in ["position","points"]:
            if col in st: st["prev_"+prefix+"_standing_"+col]=st.groupby([entity,"year"])[col].shift(1)
        wanted=[c for c in ["raceId",entity,"prev_"+prefix+"_standing_position","prev_"+prefix+"_standing_points"] if c in st]
        model=model.merge(st[wanted],on=["raceId",entity],how="left",validate="many_to_one")
    else:
        model["prev_"+prefix+"_standing_position"]=np.nan; model["prev_"+prefix+"_standing_points"]=np.nan
model["era"]=pd.cut(model.year,bins=[-np.inf,2013,2021,np.inf],labels=["pre-2014","2014-2021","2022+"]).astype(str)
feature_groups={
 "grid/current-event pre-race":["grid","grid_zero_or_unknown","qualifying_available"],
 "qualifying":[c for c in ["position","q1_seconds","q2_seconds","q3_seconds","best_qualifying_seconds","qualifying_gap_to_pole"] if c in model],
 "historical driver form":[c for c in model if c.startswith("driver_prior_")],
 "historical constructor form":[c for c in model if c.startswith("constructor_prior_")],
 "previous-race standings":[c for c in model if c.startswith("prev_driver_standing_") or c.startswith("prev_constructor_standing_")],
 "context":[c for c in ["year","round","circuitId","country","era","is_home_race","field_size","driverId","nationality_driver","constructorId","nationality_constructor"] if c in model]}
feature_cols=list(dict.fromkeys(sum(feature_groups.values(),[])))
forbidden_pat=re.compile(r"positionorder|resultid|points$|^status$|statusid|\blaps?\b|fastest|pit.?stop|lap.?time|\btime\b|current.*standing",re.I)
forbidden=[c for c in feature_cols if forbidden_pat.search(c) and not (c.startswith("prev_") and c.endswith("_points"))]
assert not forbidden, f"Forbidden post-race/current-event columns in features: {forbidden}"
assert "positionOrder" not in feature_cols and "podium" not in feature_cols
assert not model.duplicated(["raceId","driverId"]).any()
model[(["raceId","driverId","podium"]+feature_cols)].to_csv(TABLE_DIR/"modeling_table.csv",index=False)
feature_dict=[]
for group,cols in feature_groups.items():
 for c in cols: feature_dict.append({"feature":c,"group":group,"definition":c.replace("_"," "),"source_tables":"results + prior races; metadata tables as applicable","availability":"pre-race / historical only","leakage_safe_justification":"current event labels/outcomes are excluded; history shifted by one prior event","missing_data_treatment":"median/most-frequent or explicit category in train-fitted pipeline"})
feature_dict=pd.DataFrame(feature_dict); feature_dict.to_csv(TABLE_DIR/"feature_dictionary.csv",index=False); display(feature_dict)
core_missing=model[[c for c in feature_cols if c in model]].isna().mean().sort_values(ascending=False).rename("missing_fraction").reset_index().rename(columns={"index":"feature"})
core_missing.to_csv(TABLE_DIR/"model_feature_missingness.csv",index=False); display(core_missing)
fig,ax=plt.subplots(figsize=(10,max(4,.28*len(core_missing)))); sns.barplot(data=core_missing,x="missing_fraction",y="feature",color="#72b7b2",ax=ax); ax.set(title="Missingness in candidate pre-race features",xlabel="Fraction missing",ylabel="Feature",xlim=(0,1)); savefig("model_feature_missingness.png")
print("Selected feature columns:",feature_cols); print("Forbidden-column check:",forbidden)
# Spot-check: first observation per entity has no historical rate, and history columns are prior-only by construction.
display(model.groupby("driverId").head(1)[["driverId","raceId","driver_prior_race_count","driver_prior_podium_rate_expanding"]].head(8))
model.to_parquet(TABLE_DIR/"modeling_table.parquet",index=False)

   driverId  ...  driver_prior_podium_rate_expanding
0       642  ...                                 NaN
1       786  ...                                 NaN
2       686  ...                                 NaN
3       704  ...                                 NaN
4       627  ...                                 NaN

[5 rows x 4 columns]
    constructorId  ...  constructor_prior_podium_rate_expanding
0              51  ...                                      NaN
3             154  ...                                      NaN
5             151  ...                                      NaN
8             105  ...                                      NaN
13            126  ...                                      NaN

[5 rows x 4 columns]
                                    feature  ...                             missing_data_treatment
0                                      grid  ...  median/most-frequent or explicit category in t...
1                      grid_zero_or_unknown  ...  medi

notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 6 — Feature evidence, chronological split, and preprocessing
**Purpose:** evaluate feature associations and establish an honest time split. **Method:** use the preferred 2019/2020–21/2022–24 windows when possible; otherwise derive contiguous year windows and report them. The training-only ColumnTransformer imputes, indicates missingness, scales numeric data, and one-hot encodes categories. **Assumptions:** historical years are present in ascending order. **Leakage risk:** transformations fit only on training rows; validation/test are never resampled. **Interpretation:** validation drives selection; test is reserved for a single final comparison.

In [16]:
numeric_features=[c for c in feature_cols if pd.api.types.is_numeric_dtype(model[c]) and c not in {"driverId","constructorId","circuitId"}]
categorical_features=[c for c in feature_cols if c not in numeric_features]
for c in categorical_features: model[c]=model[c].map(lambda value:str(value) if pd.notna(value) else np.nan)
def chronological_year_split(df):
    years=sorted(pd.Series(df.year.dropna().astype(int).unique()).tolist())
    if not years: raise ValueError("No valid season years available for chronological splitting.")
    train=[y for y in years if y<=TRAIN_END_YEAR]; val=[y for y in VALIDATION_YEARS if y in years]; test=[y for y in range(TEST_YEARS[0],TEST_YEARS[1]+1) if y in years]
    if train and val and test: return train,val,test,"preferred years"
    # Closest viable contiguous split: 60/20/20 by distinct year count, with at least one year in each.
    if len(years)<3: raise ValueError(f"Need at least three distinct seasons for train/validation/test; found {years}.")
    n=len(years); a=max(1,int(np.floor(.6*n))); b=max(a+1,int(np.floor(.8*n))); return years[:a],years[a:b],years[b:],"automatic contiguous fallback"
train_years,val_years,test_years,split_note=chronological_year_split(model)
train_df=model[model.year.isin(train_years)].copy(); val_df=model[model.year.isin(val_years)].copy(); test_df=model[model.year.isin(test_years)].copy()
print({"split":split_note,"train_years":train_years,"validation_years":val_years,"test_years":test_years,"rows":(len(train_df),len(val_df),len(test_df))})
if min(len(train_df),len(val_df),len(test_df))==0: raise ValueError("Chronological split produced an empty partition. Review year coverage.")
X_train,y_train=train_df[feature_cols],train_df.podium.astype(int); X_val,y_val=val_df[feature_cols],val_df.podium.astype(int); X_test,y_test=test_df[feature_cols],test_df.podium.astype(int)
def make_preprocessor(scale=True, indicators=True):
    num=Pipeline([("imputer",SimpleImputer(strategy="median",add_indicator=indicators)),("scale",StandardScaler() if scale else "passthrough")])
    cat=Pipeline([("imputer",SimpleImputer(strategy="constant",fill_value="missing")),("onehot",OneHotEncoder(handle_unknown="ignore",min_frequency=2))])
    return ColumnTransformer([("num",num,numeric_features),("cat",cat,categorical_features)],remainder="drop",verbose_feature_names_out=True)
preprocess=make_preprocessor(); preprocess_no_scale=make_preprocessor(scale=False,indicators=False)
print("Preprocessing experiment A: median imputation + missing indicators -> scaling -> one-hot.")
print("Preprocessing experiment B: median/constant imputation without indicators or scaling (tree model).")
display(pd.DataFrame({"split":["train","validation","test"],"rows":[len(y_train),len(y_val),len(y_test)],"podium_rate":[y_train.mean(),y_val.mean(),y_test.mean()]}))
form_cols=[c for c in feature_cols if c.startswith("driver_prior_") or c.startswith("constructor_prior_")]
if form_cols:
    form_evidence=model.groupby("podium")[form_cols].mean().T.reset_index().rename(columns={"index":"feature",0:"non_podium_mean",1:"podium_mean"})
    form_evidence.to_csv(TABLE_DIR/"prior_form_by_target.csv",index=False); display(form_evidence)
    fig,ax=plt.subplots(figsize=(10,5)); sns.barplot(data=form_evidence.melt(id_vars="feature",var_name="outcome",value_name="mean"),x="mean",y="feature",hue="outcome",ax=ax); ax.set(title="Prior form features by observed podium label",xlabel="Feature mean",ylabel="Prior form feature"); savefig("prior_form_by_target.png")
numeric_evidence=[c for c in numeric_features if model[c].nunique(dropna=True)>1]
if numeric_evidence:
    corr=model[numeric_evidence+["podium"]].corr(numeric_only=True); corr.to_csv(TABLE_DIR/"numeric_feature_correlations.csv")
    plt.figure(figsize=(max(8,len(corr)*.55),7)); sns.heatmap(corr,cmap="vlag",center=0); plt.title("Numeric feature and target correlations (descriptive)"); savefig("numeric_feature_correlations.png")
fig,ax=plt.subplots(figsize=(8,4)); sns.barplot(x=["train","validation","test"],y=[y_train.mean(),y_val.mean(),y_test.mean()],ax=ax); ax.set(title="Podium prevalence by chronological split",xlabel="Partition",ylabel="Podium fraction"); savefig("split_class_prevalence.png")

{'split': 'preferred years', 'train_years': [1950, 1951, 1952, 1953, 1954, 1955, 1956, 1957, 1958, 1959, 1960, 1961, 1962, 1963, 1964, 1965, 1966, 1967, 1968, 1969, 1970, 1971, 1972, 1973, 1974, 1975, 1976, 1977, 1978, 1979, 1980, 1981, 1982, 1983, 1984, 1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019], 'validation_years': [2020, 2021], 'test_years': [2022, 2023, 2024], 'rows': (24529, 780, 1359)}
Preprocessing experiment A: median imputation + missing indicators -> scaling -> one-hot.
Preprocessing experiment B: median/constant imputation without indicators or scaling (tree model).
        split   rows  podium_rate
0       train  24529     0.125321
1  validation    780     0.150000
2        test   1359     0.150110
podium                                  feature  non_podium_mean  podium_mean
0                       driver_prio

notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 7. Comparable models and validation-only selection
**Purpose:** compare Logistic Regression, Random Forest, and a shallow neural network. **Method:** all three use identical chronological partitions and feature columns; thresholds maximize validation F1 and validation PR-AUC selects the final model. The neural network is implemented with scikit-learn's MLPClassifier so it runs without a TensorFlow-only dependency. **Assumptions:** both classes exist in each evaluated split. **Leakage risk:** no test feedback is used for selection. **Interpretation:** PR-AUC is primary because podium is a minority label; differences are descriptive, not proof of superiority.

In [18]:
from copy import deepcopy
def safe_auc(y,p,metric):
    return float(metric(y,p)) if pd.Series(y).nunique()==2 else np.nan
def best_threshold(y,p):
    pr,rc,th=precision_recall_curve(y,p)
    if len(th)==0: return .5
    f=2*pr[:-1]*rc[:-1]/np.maximum(pr[:-1]+rc[:-1],1e-12)
    return float(th[int(np.nanargmax(f))])
def metrics(y,p,t):
    pred=(np.asarray(p)>=t).astype(int)
    return {"roc_auc":safe_auc(y,p,roc_auc_score),"pr_auc":safe_auc(y,p,average_precision_score),"f1":f1_score(y,pred,zero_division=0),"precision":precision_score(y,pred,zero_division=0),"recall":recall_score(y,pred,zero_division=0),"balanced_accuracy":balanced_accuracy_score(y,pred),"confusion_matrix":confusion_matrix(y,pred).tolist()}
def fit_eval(name, estimator, Xtr,ytr,Xv,yv):
    estimator.fit(Xtr,ytr); pv=estimator.predict_proba(Xv)[:,1]; t=best_threshold(yv,pv)
    return {"name":name,"estimator":estimator,"threshold":t,"val_probability":pv,"val_metrics":metrics(yv,pv,t)}
def fit_eval_mlp(name, estimator, Xtr,ytr,Xv,yv, max_epochs=NN_EPOCHS, patience=12):
    """Fit one MLP epoch at a time; monitor external validation loss and restore best weights."""
    prep=estimator.named_steps["prep"]; mdl=estimator.named_steps["model"]
    Ztr=prep.fit_transform(Xtr,ytr); Zv=prep.transform(Xv)
    if hasattr(Ztr,"toarray"): Ztr=Ztr.toarray(); Zv=Zv.toarray()
    history=[]; best_loss=np.inf; best_model=None; best_epoch=0; stale=0
    for epoch in range(1,max_epochs+1):
        mdl.partial_fit(Ztr,np.asarray(ytr,dtype=int),classes=np.array([0,1]) if epoch==1 else None)
        pv=mdl.predict_proba(Zv)[:,1]
        vl=float(log_loss(yv,pv,labels=[0,1])); ap=float(average_precision_score(yv,pv)); ra=safe_auc(yv,pv,roc_auc_score)
        history.append({"epoch":epoch,"training_loss":float(mdl.loss_),"validation_loss":vl,"validation_pr_auc":ap,"validation_roc_auc":ra})
        if vl < best_loss-1e-5:
            best_loss=vl; best_epoch=epoch; best_model=deepcopy(mdl); stale=0
        else:
            stale+=1
        if stale>=patience: break
    if best_model is None: raise RuntimeError("MLP did not produce a checkpoint.")
    estimator.steps[0]=("prep",prep); estimator.steps[1]=( "model",best_model )
    pv=estimator.predict_proba(Xv)[:,1]; threshold=best_threshold(yv,pv)
    return {"name":name,"estimator":estimator,"threshold":threshold,"val_probability":pv,"val_metrics":metrics(yv,pv,threshold),"history":pd.DataFrame(history),"best_epoch":best_epoch}
models=[]
lr=Pipeline([("prep",make_preprocessor()),("model",LogisticRegression(max_iter=1500,class_weight="balanced",C=0.5,random_state=SEED))])
rf=Pipeline([("prep",make_preprocessor(scale=False,indicators=True)),("model",RandomForestClassifier(n_estimators=RF_TREES,max_depth=RF_MAX_DEPTH,min_samples_leaf=3,class_weight="balanced_subsample",random_state=SEED,n_jobs=1))])

In [19]:
try:
    lr_result=fit_eval("Logistic Regression",lr,X_train,y_train,X_val,y_val); models.append(lr_result); print("Logistic Regression validation",lr_result["val_metrics"],"threshold",lr_result["threshold"])
except Exception as exc: print(f"Logistic Regression unavailable for these data: {exc}")

Logistic Regression validation {'roc_auc': 0.9228577690116151, 'pr_auc': 0.6795413701177199, 'f1': 0.7012987012987013, 'precision': 0.7105263157894737, 'recall': 0.6923076923076923, 'balanced_accuracy': 0.8212669683257918, 'confusion_matrix': [[630, 33], [36, 81]]} threshold 0.8217732999169987


In [20]:
try:
    rf_result=fit_eval("Random Forest",rf,X_train,y_train,X_val,y_val); models.append(rf_result); print("Random Forest validation",rf_result["val_metrics"],"threshold",rf_result["threshold"])
except Exception as exc: print(f"Random Forest unavailable for these data: {exc}")

Random Forest validation {'roc_auc': 0.9182426422245427, 'pr_auc': 0.6967385388973657, 'f1': 0.7043478260869566, 'precision': 0.7168141592920354, 'recall': 0.6923076923076923, 'balanced_accuracy': 0.8220211161387632, 'confusion_matrix': [[631, 32], [36, 81]]} threshold 0.7499779855279057


In [21]:
# TensorFlow was unavailable in this execution environment; use the requested shallow neural-net fallback in scikit-learn.
# A manual epoch loop monitors external validation log loss and restores the best validation-loss weights.
nn_model=Pipeline([
    ("prep",make_preprocessor(scale=True,indicators=True)),
    ("model",MLPClassifier(hidden_layer_sizes=(64,32),activation="relu",solver="adam",
        alpha=0.001,batch_size=NN_BATCH_SIZE,learning_rate_init=0.001,max_iter=1,
        early_stopping=False,random_state=SEED,verbose=False))
])
nn_result=None
try:
    nn_result=fit_eval_mlp("Shallow Neural Network",nn_model,X_train,y_train,X_val,y_val,max_epochs=NN_EPOCHS,patience=12)
    models.append(nn_result)
    nn_history=nn_result["history"]
    nn_history.to_csv(TABLE_DIR/"neural_network_history.csv",index=False)
    fig,ax=plt.subplots(figsize=(9,4)); ax.plot(nn_history.epoch,nn_history.training_loss,label="training loss"); ax.plot(nn_history.epoch,nn_history.validation_loss,label="validation loss"); ax.axvline(nn_result["best_epoch"],color="gray",linestyle="--",label="best epoch")
    ax.set(title="Shallow neural network loss by epoch",xlabel="Epoch",ylabel="Binary log loss"); ax.legend(); savefig("nn_loss.png")
    fig,ax=plt.subplots(figsize=(9,4)); ax.plot(nn_history.epoch,nn_history.validation_pr_auc,label="validation PR-AUC"); ax.plot(nn_history.epoch,nn_history.validation_roc_auc,label="validation ROC-AUC"); ax.axvline(nn_result["best_epoch"],color="gray",linestyle="--",label="best-loss epoch")
    ax.set(title="Shallow neural network validation metrics by epoch",xlabel="Epoch",ylabel="Score"); ax.legend(); savefig("nn_validation_metrics.png")
    print("Shallow Neural Network validation",nn_result["val_metrics"],"threshold",nn_result["threshold"],"epochs",len(nn_history),"best epoch by validation loss",nn_result["best_epoch"])
except Exception as exc:
    print(f"Shallow neural network could not be trained: {type(exc).__name__}: {exc}")

Shallow Neural Network validation {'roc_auc': 0.9239535393381547, 'pr_auc': 0.7338965429840085, 'f1': 0.7, 'precision': 0.6829268292682927, 'recall': 0.717948717948718, 'balanced_accuracy': 0.8295625942684766, 'confusion_matrix': [[624, 39], [33, 84]]} threshold 0.45591477999849345 epochs 17 best epoch by validation loss 5


notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


In [22]:
if not models: raise RuntimeError("No model could be fitted. Inspect input columns and class counts.")
best=max(models,key=lambda r:r["val_metrics"]["pr_auc"] if pd.notna(r["val_metrics"]["pr_auc"]) else -np.inf)
print("Selected using validation PR-AUC:",best["name"],"threshold frozen:",best["threshold"])
pd.DataFrame([{"model":m["name"],**{k:v for k,v in m["val_metrics"].items() if k!="confusion_matrix"},"threshold":m["threshold"]} for m in models]).to_csv(TABLE_DIR/"validation_model_comparison.csv",index=False)

Selected using validation PR-AUC: Shallow Neural Network threshold frozen: 0.45591477999849345


In [23]:
# Controlled preprocessing-order experiment: same LR family, same rows/features/split, validation metrics only.
preprocessing_experiments=[]
for exp_name, prep in [("impute-indicator-scale-encode",make_preprocessor(scale=True,indicators=True)),("impute-no-indicator-no-scale-encode",make_preprocessor(scale=False,indicators=False))]:
    try:
        exp_model=Pipeline([("prep",prep),("model",LogisticRegression(max_iter=1500,class_weight="balanced",C=.5,random_state=SEED))])
        exp_model.fit(X_train,y_train); ep=exp_model.predict_proba(X_val)[:,1]; et=best_threshold(y_val,ep); preprocessing_experiments.append({"experiment":exp_name,"threshold":et,**metrics(y_val,ep,et)})
    except Exception as exc: preprocessing_experiments.append({"experiment":exp_name,"error":str(exc)})
prep_exp=pd.DataFrame(preprocessing_experiments); prep_exp.to_csv(TABLE_DIR/"preprocessing_experiments.csv",index=False); display(prep_exp)
import joblib
artifact={"model_name":best["name"],"threshold":best["threshold"],"feature_columns":feature_cols,"numeric_features":numeric_features,"categorical_features":categorical_features,"estimator":best["estimator"]}
artifact_path=ARTIFACT_DIR/"selected_model.joblib"
joblib.dump(artifact,artifact_path)
print("Serialized selected model and frozen threshold to",artifact_path)

                            experiment  ...       confusion_matrix
0        impute-indicator-scale-encode  ...  [[630, 33], [36, 81]]
1  impute-no-indicator-no-scale-encode  ...  [[631, 32], [35, 82]]

[2 rows x 9 columns]
Serialized selected model and frozen threshold to outputs\artifacts\selected_model.joblib


C:\Users\SPECTRE  X360  -14\Documents\Codex\2026-10-14\i-a\work\vendor\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## 8 — Final evaluation on train, validation, and untouched test
**Purpose:** report discrimination and thresholded performance. **Method:** freeze each validation-selected threshold, then evaluate all partitions; plot test ROC and precision-recall curves. **Assumptions:** test labels are available for retrospective evaluation. **Leakage risk:** test scores are reported once and do not influence model or threshold. **Interpretation:** uncertainty is substantial for small test windows and shifts in rules/field strength; PR-AUC is primary and F1 is secondary.

In [25]:
def probability_for(m,X):
    return m["estimator"].predict_proba(X)[:,1]
eval_rows=[]; test_probs={}
for m in models:
    for split,X,y in [("train",X_train,y_train),("validation",X_val,y_val),("test",X_test,y_test)]:
        p=probability_for(m,X); met=metrics(y,p,m["threshold"])
        eval_rows.append({"model":m["name"],"split":split,"threshold":m["threshold"],**{k:v for k,v in met.items() if k!="confusion_matrix"},"confusion_matrix":str(met["confusion_matrix"])})
        if split=="test": test_probs[m["name"]]=p
eval_table=pd.DataFrame(eval_rows); display(eval_table); eval_table.to_csv(TABLE_DIR/"model_evaluation.csv",index=False)
fig,ax=plt.subplots(figsize=(7,6))
for m in models:
    p=test_probs[m["name"]]
    if y_test.nunique()==2:
        fpr,tpr,_=roc_curve(y_test,p); ax.plot(fpr,tpr,label=f"{m['name']} AUC={roc_auc_score(y_test,p):.3f}")
ax.plot([0,1],[0,1],"--",color="gray"); ax.set(title="Test ROC curves",xlabel="False positive rate",ylabel="True positive rate"); ax.legend(); savefig("test_roc_curves.png")
fig,ax=plt.subplots(figsize=(7,6))
for m in models:
    p=test_probs[m["name"]]; pr,rc,_=precision_recall_curve(y_test,p); ax.plot(rc,pr,label=f"{m['name']} AP={average_precision_score(y_test,p):.3f}")
ax.set(title="Test precision-recall curves",xlabel="Recall",ylabel="Precision"); ax.legend(); savefig("test_pr_curves.png")
print("Confusion matrices are serialized by split and model in the evaluation table. Test set was not used for selection.")

                    model  ...               confusion_matrix
0     Logistic Regression  ...  [[20091, 1364], [1316, 1758]]
1     Logistic Regression  ...          [[630, 33], [36, 81]]
2     Logistic Regression  ...       [[1043, 112], [53, 151]]
3           Random Forest  ...  [[20209, 1246], [1290, 1784]]
4           Random Forest  ...          [[631, 32], [36, 81]]
5           Random Forest  ...        [[1073, 82], [77, 127]]
6  Shallow Neural Network  ...   [[20542, 913], [1523, 1551]]
7  Shallow Neural Network  ...          [[624, 39], [33, 84]]
8  Shallow Neural Network  ...       [[1020, 135], [43, 161]]

[9 rows x 10 columns]
Confusion matrices are serialized by split and model in the evaluation table. Test set was not used for selection.


notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 9 — Ablation experiments
**Purpose:** estimate how much validation predictive performance changes when feature groups are omitted. **Method:** retrain the selected model family with one group removed; choose each threshold on validation. **Assumptions:** grouped features and partitions are unchanged. **Leakage risk:** validation is used for model comparison only; no test tuning. **Interpretation:** ablation measures predictive reliance in this dataset, not a causal effect.

In [27]:
ablation=[]
for group,cols in feature_groups.items():
    reduced=[c for c in feature_cols if c not in cols]
    if not reduced: continue
    old_features,old_num,old_cat=feature_cols,numeric_features,categorical_features
    feature_cols=reduced; numeric_features=[c for c in reduced if c in old_num]; categorical_features=[c for c in reduced if c in old_cat]
    try:
        if best["name"]=="Logistic Regression": est=Pipeline([("prep",make_preprocessor()),("model",LogisticRegression(max_iter=1500,class_weight="balanced",C=.5,random_state=SEED))])
        elif best["name"]=="Random Forest": est=Pipeline([("prep",make_preprocessor(scale=False)),("model",RandomForestClassifier(n_estimators=RF_TREES,max_depth=RF_MAX_DEPTH,min_samples_leaf=3,class_weight="balanced_subsample",random_state=SEED,n_jobs=1))])
        else:
            est=Pipeline([("prep",make_preprocessor(scale=True,indicators=True)),("model",MLPClassifier(hidden_layer_sizes=(64,32),activation="relu",solver="adam",alpha=.001,batch_size=NN_BATCH_SIZE,learning_rate_init=.001,max_iter=1,early_stopping=False,random_state=SEED))])
        rr=(fit_eval_mlp("ablation",est,X_train[reduced],y_train,X_val[reduced],y_val,max_epochs=NN_EPOCHS,patience=12) if best["name"]=="Shallow Neural Network" else fit_eval("ablation",est,X_train[reduced],y_train,X_val[reduced],y_val)); ablation.append({"removed_group":group,"features_removed":len(cols),**rr["val_metrics"],"threshold":rr["threshold"]})
    except Exception as exc: ablation.append({"removed_group":group,"error":str(exc)})
    finally: feature_cols,numeric_features,categorical_features=old_features,old_num,old_cat
full=best["val_metrics"]; ab=pd.DataFrame(ablation)
for m in ["pr_auc","roc_auc","f1","precision","recall"]:
    if m in ab: ab[m+"_difference_from_full"]=ab[m]-full.get(m,np.nan)
ab.to_csv(TABLE_DIR/"ablation_results.csv",index=False); display(ab)
if "pr_auc" in ab:
    fig,ax=plt.subplots(figsize=(10,5)); sns.barplot(data=ab,x="pr_auc_difference_from_full",y="removed_group",ax=ax,color="#72b7b2"); ax.axvline(0,color="black",lw=1); ax.set(title="Validation PR-AUC change after removing feature group",xlabel="PR-AUC difference from full model",ylabel="Removed feature group"); savefig("ablation_pr_auc.png")

                 removed_group  ...  recall_difference_from_full
0  grid/current-event pre-race  ...                    -0.008547
1                   qualifying  ...                    -0.042735
2       historical driver form  ...                    -0.042735
3  historical constructor form  ...                     0.008547
4      previous-race standings  ...                    -0.068376
5                      context  ...                     0.111111

[6 rows x 15 columns]


notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 10 — Explainability
**Purpose:** inspect global and local model behavior. **Method:** validation permutation importance; SHAP where a compatible explainer is available; local SHAP or a clearly reported fallback. **Assumptions:** explanations describe the fitted selected model. **Leakage risk:** validation features only; labels are shown for context. **Interpretation:** Feature importance and SHAP/LIME values describe how the model used associations in this dataset; they do not establish that a feature causes a podium finish.

In [29]:
# Readable global permutation importance on the untouched validation features.
pi=permutation_importance(best["estimator"],X_val,y_val,n_repeats=5,random_state=SEED,scoring="average_precision",n_jobs=1)
pitab=pd.DataFrame({"feature":feature_cols,"importance_mean":pi.importances_mean,"importance_std":pi.importances_std}).sort_values("importance_mean",ascending=False)
pitab.to_csv(TABLE_DIR/"permutation_importance.csv",index=False); display(pitab.head(20))
plt.figure(figsize=(9,6)); sns.barplot(data=pitab.head(20),x="importance_mean",y="feature",color="#f2cf5b"); plt.title("Validation permutation importance (average precision)"); plt.xlabel("Mean decrease in average precision"); plt.ylabel("Feature"); savefig("permutation_importance.png")

# Explain class-1 probability in transformed feature space. Use exact model-specific explainers
# for the two statistical baselines and a bounded permutation explainer for the MLP.
prep=best["estimator"].named_steps["prep"]; mdl=best["estimator"].named_steps["model"]
feature_names=np.asarray(prep.get_feature_names_out(),dtype=str)
background_raw=X_train.sample(n=min(40,len(X_train)),random_state=SEED)
explain_raw=X_val.sample(n=min(30,len(X_val)),random_state=SEED)
Z_background=prep.transform(background_raw); Z_explain=prep.transform(explain_raw)
if hasattr(Z_background,"toarray"): Z_background=Z_background.toarray(); Z_explain=Z_explain.toarray()
if SHAP_AVAILABLE:
    try:
        if isinstance(mdl,RandomForestClassifier):
            explainer=shap.TreeExplainer(mdl,data=Z_background[:20],feature_perturbation="interventional",model_output="probability")
            shap_values=explainer(Z_explain[:20],check_additivity=False)
        elif isinstance(mdl,LogisticRegression):
            explainer=shap.LinearExplainer(mdl,Z_background)
            shap_values=explainer(Z_explain)
        else:
            predict_transformed=lambda z: mdl.predict_proba(z)[:,1]
            explainer=shap.PermutationExplainer(predict_transformed,Z_background[:10],feature_names=feature_names)
            shap_values=explainer(Z_explain[:12],max_evals=2*Z_explain.shape[1]+1)
        values=np.asarray(shap_values.values)
        if values.ndim==3: values=values[:,:,1]
        shap.summary_plot(values,Z_explain[:len(values)],feature_names=feature_names,show=False,max_display=20)
        plt.title("Global SHAP summary on validation rows"); savefig("shap_summary.png")
        shap_importance=pd.DataFrame({"feature":feature_names,"mean_absolute_shap":np.abs(values).mean(axis=0)}).sort_values("mean_absolute_shap",ascending=False)
        shap_importance.to_csv(TABLE_DIR/"shap_global_importance.csv",index=False); display(shap_importance.head(20))
        print("SHAP global explanation rows:",len(values),"transformed feature count:",len(feature_names))
    except Exception as exc:
        shap_values=None; print(f"Global SHAP explanation failed: {type(exc).__name__}: {exc}")
else:
    shap_values=None; print("SHAP unavailable; install shap and rerun this notebook.")

# Local SHAP for one actual held-out test prediction: choose the most confident test row.
idx=int(np.argmax(np.abs(np.asarray(test_probs[best["name"]])-y_test.to_numpy()))) if len(y_test) else 0
local_row=X_test.iloc[[idx]]; local_probability=float(test_probs[best["name"]][idx]); actual=int(y_test.iloc[idx])
display(local_row.T.rename(columns={local_row.index[0]:"pre-race values"})); print({"predicted_probability":local_probability,"threshold":best["threshold"],"predicted_class":int(local_probability>=best["threshold"]),"actual":actual,"row_index":int(local_row.index[0])})
if SHAP_AVAILABLE:
    try:
        Z_local=prep.transform(local_row)
        if hasattr(Z_local,"toarray"): Z_local=Z_local.toarray()
        if isinstance(mdl,RandomForestClassifier):
            local_explainer=shap.TreeExplainer(mdl,data=Z_background[:20],feature_perturbation="interventional",model_output="probability")
            local_values=local_explainer(Z_local,check_additivity=False)
        elif isinstance(mdl,LogisticRegression):
            local_explainer=shap.LinearExplainer(mdl,Z_background)
            local_values=local_explainer(Z_local)
        else:
            predict_transformed=lambda z: mdl.predict_proba(z)[:,1]
            local_explainer=shap.PermutationExplainer(predict_transformed,Z_background[:10],feature_names=feature_names)
            local_values=local_explainer(Z_local,max_evals=2*Z_local.shape[1]+1)
        local_array=np.asarray(local_values.values)
        if local_array.ndim==3: local_array=local_array[:,:,1]
        local_exp=shap.Explanation(values=local_array[0],base_values=np.asarray(local_values.base_values).reshape(-1)[0],data=Z_local[0],feature_names=feature_names)
        shap.plots.waterfall(local_exp,max_display=15,show=False); plt.title("Local SHAP explanation for held-out test prediction"); savefig("local_shap_waterfall.png")
        pd.DataFrame({"feature":feature_names,"shap_value":local_array[0],"feature_value":Z_local[0]}).assign(abs_shap=lambda d:d.shap_value.abs()).sort_values("abs_shap",ascending=False).to_csv(TABLE_DIR/"shap_local_explanation.csv",index=False)
    except Exception as exc: print(f"Local SHAP explanation failed: {type(exc).__name__}: {exc}")
print("Feature importance and SHAP values describe model associations in this dataset; they do not establish that a feature causes a podium finish.")

                                feature  importance_mean  importance_std
0                                  grid         0.133478        0.007971
19          prev_driver_standing_points         0.025107        0.017273
23                                round         0.024767        0.004401
4                            q2_seconds         0.024389        0.011757
7                qualifying_gap_to_pole         0.024378        0.009839
5                            q3_seconds         0.019568        0.003151
1                  grid_zero_or_unknown         0.016679        0.005302
29                             driverId         0.011992        0.016505
24                            circuitId         0.010233        0.004013
30                   nationality_driver         0.009632        0.004840
17  constructor_prior_podium_rate_last5         0.005896        0.007193
9    driver_prior_podium_rate_expanding         0.004978        0.002903
12       driver_prior_podium_rate_last5         0.0

notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_28:29: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
notebook_cell_8:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## 11 — Inference function
**Purpose:** provide a reusable one-record inference interface. **Method:** accept pre-race fields, enforce required fields, fill optional omissions through the fitted pipeline, and return probability/class/threshold/model/warnings. **Assumptions:** the record's keys match the feature names and include grid position. **Leakage risk:** post-race keys are explicitly rejected. **Interpretation:** unknown categories are supported by one-hot `handle_unknown='ignore'`; records for novel drivers/teams have no learned identity-specific history.

In [31]:
POST_RACE_KEYS={"positionOrder","podium","points","status","statusId","laps","time","milliseconds","fastestLap","fastestLapTime","fastestLapSpeed","rank","pitStops","lapTimes"}
REQUIRED_INFERENCE=[c for c in ["grid"] if c in feature_cols]
OPTIONAL_INFERENCE=[c for c in feature_cols if c not in REQUIRED_INFERENCE]
def predict_podium_probability(record:dict)->dict:
    if not isinstance(record,dict): raise TypeError("record must be a dictionary of pre-race fields.")
    leaked=POST_RACE_KEYS.intersection(record)
    if leaked: raise ValueError(f"Post-race fields are prohibited at inference: {sorted(leaked)}")
    missing_required=[c for c in REQUIRED_INFERENCE if c not in record or record[c] is None]
    if missing_required: raise ValueError(f"Missing required pre-race fields: {missing_required}")
    missing_optional=[c for c in OPTIONAL_INFERENCE if c not in record or record[c] is None]
    row=pd.DataFrame([{c:record.get(c,np.nan) for c in feature_cols}])
    for c in categorical_features: row[c]=row[c].map(lambda value:str(value) if pd.notna(value) else np.nan)
    warnings_list=[]
    if missing_optional: warnings_list.append("Optional fields missing and imputed: "+", ".join(missing_optional))
    p=float(probability_for(best,row)[0]); prediction=int(p>=best["threshold"])
    return {"predicted_probability":p,"predicted_class":prediction,"threshold":float(best["threshold"]),"model_name":best["name"],"warnings":warnings_list}
demo={c:(float(X_train[c].median()) if c in numeric_features else (X_train[c].mode(dropna=True).iloc[0] if X_train[c].notna().any() else "unseen")) for c in feature_cols}
demo["grid"]=1 if "grid" in feature_cols else None
print("Complete-record demonstration:",predict_podium_probability(demo))
partial={c:demo[c] for c in REQUIRED_INFERENCE}
if categorical_features: partial[categorical_features[0]]="unseen-category-demo"
print("Partially missing / unknown-category demonstration:",predict_podium_probability(partial))

Complete-record demonstration: {'predicted_probability': 0.43072846189772546, 'predicted_class': 0, 'threshold': 0.45591477999849345, 'model_name': 'Shallow Neural Network', 'warnings': []}
Partially missing / unknown-category demonstration: {'predicted_probability': 0.5034200365095294, 'predicted_class': 1, 'threshold': 0.45591477999849345, 'model_name': 'Shallow Neural Network', 'warnings': ['Optional fields missing and imputed: grid_zero_or_unknown, qualifying_available, q1_seconds, q2_seconds, q3_seconds, best_qualifying_seconds, qualifying_gap_to_pole, driver_prior_race_count, driver_prior_podium_rate_expanding, driver_prior_finish_mean_expanding, driver_prior_podium_rate_last3, driver_prior_podium_rate_last5, constructor_prior_race_count, constructor_prior_podium_rate_expanding, constructor_prior_finish_mean_expanding, constructor_prior_podium_rate_last3, constructor_prior_podium_rate_last5, prev_driver_standing_position, prev_driver_standing_points, prev_constructor_standing_pos

## 12 — Final checklist, limitations, and generated files
**Purpose:** make deliverables auditable. **Method:** verify core table grain and list files created. **Assumptions:** the notebook completed all sections without a missing-data stop. **Leakage risk:** checklist refers to implemented checks. **Interpretation:** status is evidence-based; sections requiring optional dependencies are marked based on runtime availability.

In [33]:
checklist=[
 ("Recursive CSV discovery and paths","Section 1","data_inventory.csv","PASS","actual paths printed; missing core tables stop with instructions"),
 ("Pre-cleaning visual EDA","Section 1","raw_entries_before_cleaning.png; raw_missingness_before_cleaning.png; raw_duplicate_keys_before_cleaning.png","PASS","visualizes raw loaded data before type conversion/duplicate resolution"),
 ("Semantic date and duration typing","Section 2","cleaning_log.csv; lap_times.time_seconds","PASS","qualifying and lap durations parsed; original strings retained"),
 ("Key and foreign-key audit","Section 2","key_audit.csv; foreign_key_audit.csv","PASS","findings computed from discovered data"),
 ("Duplicate race-driver handling","Section 2","cleaning_log.csv; duplicate_rows_by_year.csv","PASS","deterministic rule; unique-grain assertion"),
 ("Before/after cleaning EDA","Section 3","cleaning_impact_by_season.png; cleaning_impact_by_season.csv","PASS","paired raw and cleaned row counts by season"),
 ("Three engineering questions","Section 4","q1/q2/q3 tables and figures","PASS","rates, figures, grid-versus-qualifying audit, Indianapolis sensitivity"),
 ("Leakage-safe features and dictionary","Section 5","modeling_table.csv/.parquet; feature_dictionary.csv","PASS","forbidden-feature assertion and shifted history"),
 ("Chronological splits and preprocessing","Section 6","split_class_prevalence.png","PASS","train-only fit; partitions printed"),
 ("Three model families","Section 7","validation_model_comparison.csv","PASS","Logistic Regression, Random Forest, shallow sklearn MLP fallback"),
 ("Neural-network epoch diagnostics","Section 7","neural_network_history.csv; nn_loss.png; nn_validation_metrics.png","PASS","train/validation loss and PR-AUC/ROC-AUC by epoch; best val-loss weights restored"),
 ("Test evaluation","Section 8","model_evaluation.csv; ROC/PR figures","PASS","thresholds selected on validation"),
 ("Ablation and preprocessing experiments","Sections 7-9","ablation_results.csv; preprocessing_experiments.csv","PASS","validation-only comparisons"),
 ("Explainability","Section 10","permutation/SHAP plots","PASS","global permutation, global SHAP, and local SHAP"),
 ("Inference demonstrations","Section 11","notebook output","PASS","required/optional fields and forbidden keys handled"),
 ("Analytical report","companion files","analytical_report.pdf","PASS","completed from generated tables and plots")]
check=pd.DataFrame(checklist,columns=["requirement","notebook_section","generated_evidence","status","notes"]); display(check); check.to_csv(TABLE_DIR/"final_checklist.csv",index=False)
limits=["Qualifying coverage changes over historical seasons.","F1 rules, cars, calendars, and field strength change across eras.","Some circuit and constructor samples are small.","Indianapolis 500 (1950-1960) had a distinct oval/entry context; Q2 includes a sensitivity excluding it.","Podium outcomes are imbalanced.","Status labels are ambiguous for some mechanical retirements.","Driver and constructor identity changes limit generalization.","Associational explanations and home-race comparisons do not establish causality.","Source files may contain missing values, duplicated snapshots, or inconsistent labels."]
display(pd.DataFrame({"limitation":limits}))
created=sorted(str(p) for p in OUTPUT_ROOT.rglob("*") if p.is_file()); print("Generated files (current run):"); print("\n".join(created)); print("Run: jupyter lab f1_podium_prediction_milestone1.ipynb; place Ergast CSVs under DATA_ROOT first.")

                               requirement  ...                                              notes
0        Recursive CSV discovery and paths  ...  actual paths printed; missing core tables stop...
1                  Pre-cleaning visual EDA  ...  visualizes raw loaded data before type convers...
2        Semantic date and duration typing  ...  qualifying and lap durations parsed; original ...
3                Key and foreign-key audit  ...             findings computed from discovered data
4           Duplicate race-driver handling  ...         deterministic rule; unique-grain assertion
5                Before/after cleaning EDA  ...        paired raw and cleaned row counts by season
6              Three engineering questions  ...  rates, figures, grid-versus-qualifying audit, ...
7     Leakage-safe features and dictionary  ...    forbidden-feature assertion and shifted history
8   Chronological splits and preprocessing  ...                 train-only fit; partitions printed
9         